<a href="https://colab.research.google.com/github/jayjanii/sentiment-analysis/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
from datasets import load_dataset

In [36]:
dataset = load_dataset('imdb')

In [37]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [38]:
from transformers import BasicTokenizer
tokenizer = BasicTokenizer(do_lower_case=True)

In [39]:
tokenizer.tokenize("The movie was AMAZING!!")

['the', 'movie', 'was', 'amazing', '!', '!']

In [40]:
def build_vocabulary(dataset):
    vocab = {"<pad>": 0, "<unk>": 1}
    id = 2
    for data in dataset['train']:
        token_list = tokenizer.tokenize(data['text'])
        for token in token_list:
            if token not in vocab:
                vocab[token] = id
                id += 1

    return vocab

In [41]:
vocab = build_vocabulary(dataset)
print(len(vocab))

74878


In [59]:
import torch

def encode_review(text, vocab, max_length=256):
    tokenized_text = tokenizer.tokenize(text)
    tokenized_text = tokenized_text[:max_length]

    for i, token in enumerate(tokenized_text, start=0):
        if token in vocab:
            tokenized_text[i] = vocab[token] # vocab ID
        else:
            tokenized_text[i] = vocab["<unk>"] # not in vocab

    if len(tokenized_text) < max_length:
        n = max_length - len(tokenized_text)
        tokenized_text.extend([vocab["<pad>"]] * n) # padding

    tensor = torch.tensor(tokenized_text)

    return tensor

In [62]:
tensor = encode_review(dataset['train'][0]['text'], vocab)
print(tensor.shape)
print(tensor[:10])

torch.Size([256])
tensor([ 2,  3,  2,  4,  5,  6,  7,  8,  9, 10])


In [66]:
from torch.utils.data import Dataset

class IMDBDataset(Dataset):
    def __init__(self, data, vocab, max_length=256):
        self.data = data
        self.vocab = vocab
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoded_review = encode_review(self.data[idx]['text'], self.vocab, self.max_length)
        review_label = self.data[idx]['label']
        return (encoded_review, review_label)

In [68]:
from torch.utils.data import DataLoader

train_loader = DataLoader(IMDBDataset(dataset['train'], vocab), batch_size=32, shuffle=True)
test_loader = DataLoader(IMDBDataset(dataset['test'], vocab), batch_size=32, shuffle=False)